# Proyecto II: Detección de anomalías industriales mediante destilado de modelos y autoencoders

**Curso:** Inteligencia Artificial, Escuela de Ingeniería en Computación, Instituto Tecnológico de Costa Rica

**Integrantes:**
- Sebastián Chacón - 2023221908
- Daniel Pulido - 2019196502
- Sebastián Muñoz - 2023090681

**Notebook 4 de 4: detección de anomalías.** Usar los embeddings de los mejores checkpoints de los tres modelos para detectar anomalías con la distancia de Mahalanobis a la distribución normal de cada categoría, comparar los modelos y analizar el mejor con PCA, t-SNE y DBSCAN.

El proyecto se divide en cuatro notebooks que comparten la configuración de `conf/` y el código de `src/`:
1. `datos.ipynb`: exploración de MVTec AD y `LightningDataModule`.
2. `modelos_ab.ipynb`: modelo A (CNN desde cero) y modelo B (CNN destilada).
3. `modelo_c.ipynb`: modelo C (autoencoder U-Net).
4. `anomalias.ipynb`: detección de anomalías con embeddings y análisis con DBSCAN.

> **Placeholder.** Se desarrolla cuando los tres modelos estén entrenados.

## Contenido
1. Configuración y entorno
2. Extracción de embeddings
3. Distribución normal por categoría
4. Detección con distancia de Mahalanobis
5. Comparación de los modelos
6. Análisis con DBSCAN, PCA y t-SNE

## 1. Configuración y entorno

### 1.1 Imports y configuración con Hydra
Las clases (DataModule, modelos y utilidades de entrenamiento) están en `src/` como scripts auxiliares, y este notebook las importa y ejecuta. Para eso se agrega la raíz del repositorio a `sys.path`. Con `autoreload` los cambios en `src/` se cargan sin reiniciar el kernel.

El decorador `@hydra.main` está pensado para scripts que se ejecutan desde la terminal y no funciona dentro de Jupyter. Por eso `load_config` usa la API de composición de Hydra (`initialize_config_dir` y `compose`), que es la que recomienda su documentación para notebooks. Recibe una lista de overrides, por ejemplo `["+experiment=cnn_scratch/lr3e-4"]`, y así cada corrida varía sus hiperparámetros sin tocar los archivos YAML.

In [1]:
# Imports, repository root and Hydra configuration
%load_ext autoreload
%autoreload 2

import logging
import sys
import warnings
from pathlib import Path

# Hide noise that does not affect the results: torch's "triton not found" (no triton on Windows)
logging.getLogger("torch.utils.flop_counter").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# The notebook lives in notebooks/, the repository root is its parent folder
PROJECT_ROOT = Path.cwd().resolve().parent
assert (PROJECT_ROOT / "conf").is_dir(), "Run the notebook from the notebooks/ folder"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import wandb
from hydra.utils import instantiate
from omegaconf import OmegaConf

from src.utils import load_config, resolve_path

cfg = load_config()
print(OmegaConf.to_yaml(cfg))

data:
  _target_: src.data.MVTecDataModule
  root: data/mvtec_ad
  categories:
  - carpet
  - leather
  - tile
  - wood
  - bottle
  - cable
  - capsule
  - hazelnut
  - metal_nut
  - pill
  image_size: 224
  batch_size: 32
  num_workers: 0
  cache_dir: ${paths.output_dir}/cache
  val_split: 0.2
  seed: ${seed}
  mean:
  - 0.485
  - 0.456
  - 0.406
  std:
  - 0.229
  - 0.224
  - 0.225
model:
  _target_: src.models.cnn.CNNClassifierModule
  name: cnn_scratch
  num_classes: ${len:${data.categories}}
  head:
    hidden_dim: 256
    dropout: 0.3
  optimizer:
    _target_: torch.optim.AdamW
    _partial_: true
    lr: 0.001
    weight_decay: 0.0001
  scheduler: null
trainer:
  _target_: lightning.Trainer
  max_epochs: 50
  accelerator: auto
  devices: 1
  precision: 16-mixed
  log_every_n_steps: 10
  enable_progress_bar: false
callbacks:
  early_stopping:
    _target_: lightning.pytorch.callbacks.EarlyStopping
    monitor: val/loss
    mode: min
    patience: 5
    min_delta: 0.0
  checkpoi

## 2. Extracción de embeddings

*Pendiente.*

## 3. Distribución normal por categoría

*Pendiente.*

## 4. Detección con distancia de Mahalanobis

*Pendiente.*

## 5. Comparación de los modelos

*Pendiente.*

## 6. Análisis con DBSCAN, PCA y t-SNE

*Pendiente.*